# Maze Crawler Kaggle Starter

This notebook is set up for Kaggle's **Maze Crawler** competition: a 1v1 simulation/RL-style contest. It creates a `submission.py` agent, tries to run a local smoke test with `kaggle_environments` when the environment is available, and packages the result for submission.

Run this notebook on Kaggle after accepting the competition rules. The competition data/API is protected until then, so the first cells intentionally inspect whatever Kaggle mounts under `/kaggle/input`.

## 1. Inspect the Kaggle Runtime

In [ ]:
from pathlib import Path
import json
import os
import sys

INPUT_ROOT = Path('/kaggle/input')
WORKING = Path('/kaggle/working') if Path('/kaggle').exists() else Path.cwd()

print('Python:', sys.version)
print('Working directory:', WORKING)
print('Input root exists:', INPUT_ROOT.exists())

if INPUT_ROOT.exists():
    for path in sorted(INPUT_ROOT.rglob('*'))[:200]:
        rel = path.relative_to(INPUT_ROOT)
        suffix = '/' if path.is_dir() else f' ({path.stat().st_size:,} bytes)'
        print(f'{rel}{suffix}')
else:
    print('Not running on Kaggle, or no input directory is mounted.')

## 2. Find the Simulation Environment

Kaggle game competitions usually expose a `kaggle_environments` environment. This cell tries the most likely names. If none are available, the notebook still writes the submission agent.

In [ ]:
ENV_NAME = None
env = None

try:
    import kaggle_environments
    from kaggle_environments import make, evaluate
    print('kaggle_environments:', getattr(kaggle_environments, '__version__', 'unknown'))

    candidates = [
        'maze_crawler', 'maze-crawler', 'mazeCrawler', 'mazecrawler',
        'mab', 'lux_ai_s3', 'lux_ai_2024'
    ]
    for name in candidates:
        try:
            candidate_env = make(name, debug=True)
            ENV_NAME = name
            env = candidate_env
            print('Detected environment:', ENV_NAME)
            print('Configuration:', env.configuration)
            break
        except Exception as exc:
            print(f'No env named {name!r}: {type(exc).__name__}: {exc}')
except Exception as exc:
    print('Could not import kaggle_environments:', repr(exc))

if ENV_NAME is None:
    print('Environment was not detected in this runtime. This can be normal before Kaggle updates the package or if the competition uses a custom evaluator.')

## 3. Write a Baseline Agent

This baseline is deliberately defensive because public access to the exact observation schema is limited outside Kaggle. It handles common maze observation shapes:

- direct legal action lists
- grid/board text or arrays with player markers
- target/exit positions when exposed
- fallback deterministic exploration

After you run a few matches, improve the `choose_action` logic using the printed observations from the smoke-test section.

In [ ]:
AGENT_CODE = r'''
"""Baseline Maze Crawler agent.

Kaggle calls agent(observation, configuration) each turn. Keep this file
self-contained: the competition runner may not include notebook variables.
"""

from collections import deque
import random

DIRS = [
    ('NORTH', (0, -1)), ('EAST', (1, 0)), ('SOUTH', (0, 1)), ('WEST', (-1, 0)),
]
ALIASES = {
    'NORTH': ['NORTH', 'UP', 'N', 0],
    'EAST': ['EAST', 'RIGHT', 'E', 1],
    'SOUTH': ['SOUTH', 'DOWN', 'S', 2],
    'WEST': ['WEST', 'LEFT', 'W', 3],
}
WALL_TOKENS = {'#', 'X', 'x', 'wall', 'WALL', 'blocked', 'BLOCKED', 1}
TARGET_TOKENS = {'G', 'g', 'E', 'e', 'T', 't', '$', 'exit', 'goal', 'target'}
PLAYER_TOKENS = {'P', 'p', 'A', 'a', '@', 'player', 'agent'}

state = {
    'turn': 0,
    'last_action': None,
    'rng': random.Random(20260516),
}


def as_dict(obj):
    if isinstance(obj, dict):
        return obj
    if hasattr(obj, 'items'):
        return dict(obj.items())
    return {}


def legal_actions(obs, config):
    data = as_dict(obs)
    cfg = as_dict(config)
    for key in ('legalActions', 'legal_actions', 'validActions', 'valid_actions', 'actions'):
        value = data.get(key) or cfg.get(key)
        if value:
            return list(value)
    return ['NORTH', 'EAST', 'SOUTH', 'WEST']


def normalize_grid(raw):
    if raw is None:
        return None
    if isinstance(raw, str):
        rows = [list(line) for line in raw.splitlines() if line]
        return rows or None
    if isinstance(raw, (list, tuple)):
        rows = []
        for row in raw:
            if isinstance(row, str):
                rows.append(list(row))
            elif isinstance(row, (list, tuple)):
                rows.append(list(row))
        return rows or None
    return None


def get_grid(obs):
    data = as_dict(obs)
    for key in ('grid', 'board', 'maze', 'map', 'view', 'observation'):
        grid = normalize_grid(data.get(key))
        if grid:
            return grid
    return normalize_grid(obs)


def find_token(grid, tokens):
    if not grid:
        return None
    for y, row in enumerate(grid):
        for x, value in enumerate(row):
            if value in tokens:
                return (x, y)
    return None


def position_from_obs(obs):
    data = as_dict(obs)
    for key in ('position', 'pos', 'player', 'location'):
        value = data.get(key)
        if isinstance(value, dict) and {'x', 'y'} <= set(value):
            return (int(value['x']), int(value['y']))
        if isinstance(value, (list, tuple)) and len(value) >= 2:
            return (int(value[0]), int(value[1]))
    grid = get_grid(obs)
    found = find_token(grid, PLAYER_TOKENS)
    if found:
        return found
    if grid:
        return (len(grid[0]) // 2, len(grid) // 2)
    return None


def target_from_obs(obs):
    data = as_dict(obs)
    for key in ('target', 'goal', 'exit', 'treasure'):
        value = data.get(key)
        if isinstance(value, dict) and {'x', 'y'} <= set(value):
            return (int(value['x']), int(value['y']))
        if isinstance(value, (list, tuple)) and len(value) >= 2:
            return (int(value[0]), int(value[1]))
    return find_token(get_grid(obs), TARGET_TOKENS)


def is_open(grid, pos):
    if not grid:
        return True
    x, y = pos
    if y < 0 or y >= len(grid) or x < 0 or x >= len(grid[y]):
        return False
    return grid[y][x] not in WALL_TOKENS


def shortest_step(grid, start, target):
    if not grid or start is None or target is None:
        return None
    queue = deque([start])
    parent = {start: (None, None)}
    while queue:
        current = queue.popleft()
        if current == target:
            break
        for action, (dx, dy) in DIRS:
            nxt = (current[0] + dx, current[1] + dy)
            if nxt not in parent and is_open(grid, nxt):
                parent[nxt] = (current, action)
                queue.append(nxt)
    if target not in parent:
        return None
    cur = target
    action = None
    while parent[cur][0] is not None:
        prev, action = parent[cur]
        if prev == start:
            return action
        cur = prev
    return action


def adapt_action(action, legal):
    if action in legal:
        return action
    for alias in ALIASES.get(action, [action]):
        if alias in legal:
            return alias
        if isinstance(alias, str):
            lower = alias.lower()
            if lower in legal:
                return lower
    return legal[0] if legal else action


def choose_action(observation, configuration):
    legal = legal_actions(observation, configuration)
    grid = get_grid(observation)
    pos = position_from_obs(observation)
    target = target_from_obs(observation)

    action = shortest_step(grid, pos, target)
    if action is None and grid and pos is not None:
        # Prefer open moves, with a mild right-hand bias for exploration.
        order = ['EAST', 'SOUTH', 'WEST', 'NORTH']
        open_actions = []
        for name in order:
            dx, dy = dict(DIRS)[name]
            if is_open(grid, (pos[0] + dx, pos[1] + dy)):
                open_actions.append(name)
        if open_actions:
            action = open_actions[state['turn'] % len(open_actions)]

    if action is None:
        action = ['EAST', 'SOUTH', 'WEST', 'NORTH'][state['turn'] % 4]

    state['turn'] += 1
    state['last_action'] = action
    return adapt_action(action, legal)


def agent(observation, configuration):
    return choose_action(observation, configuration)
'''

submission_path = WORKING / 'submission.py'
submission_path.write_text(AGENT_CODE, encoding='utf-8')
print('Wrote', submission_path)
print(submission_path.read_text(encoding='utf-8')[:1200])

## 4. Smoke Test the Agent

In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location('submission', submission_path)
submission = importlib.util.module_from_spec(spec)
spec.loader.exec_module(submission)

toy_obs = {
    'grid': [
        list('#####'),
        list('#P..#'),
        list('#.#G#'),
        list('#...#'),
        list('#####'),
    ],
    'legalActions': ['NORTH', 'EAST', 'SOUTH', 'WEST'],
}
print('Toy action:', submission.agent(toy_obs, {}))

if env is not None:
    try:
        print('Running one local episode...')
        env.run([str(submission_path), str(submission_path)])
        print('Episode complete.')
        print('Final state:', env.state[-1] if env.state else 'no state')
        try:
            env.render(mode='ipython', width=900, height=700)
        except Exception as render_exc:
            print('Render skipped:', repr(render_exc))
    except Exception as run_exc:
        print('Environment smoke test failed:', repr(run_exc))
        print('If this shows an action/schema error, inspect the observation below and update submission.py.')
        try:
            print('Initial state sample:', env.reset()[0].observation)
        except Exception as obs_exc:
            print('Could not print initial observation:', repr(obs_exc))

## 5. Package for Submission

In [ ]:
import tarfile

tar_path = WORKING / 'submission.tar.gz'
with tarfile.open(tar_path, 'w:gz') as tar:
    tar.add(submission_path, arcname='submission.py')

print('Created:', submission_path)
print('Created:', tar_path)
print('Files in working directory:')
for path in sorted(WORKING.iterdir()):
    if path.is_file():
        print(f'- {path.name}: {path.stat().st_size:,} bytes')

## 6. Next Improvements

Once the smoke test exposes the exact observation schema, the highest-impact upgrades are:

- maintain a persistent explored map across turns
- use BFS/A* toward frontier cells when no target is visible
- model the opponent position and avoid direct races into traps
- run repeated self-play matches and score candidate policies

Keep `submission.py` self-contained. Kaggle's runner may not include any notebook-only variables or local helper files unless they are packaged with the submission.